# Prony Series Conversion: Time-Frequency Domain Bridging

This notebook demonstrates the Prony series conversion transform in RheoJAX, which bridges the
time domain and frequency domain representations of viscoelastic materials.

The Prony series representation is:

$$G(t) = G_e + \sum_{i=1}^{N} G_i \exp\left(-\frac{t}{\tau_i}\right)$$

with the corresponding frequency-domain expressions:

$$G'(\omega) = G_e + \sum_{i=1}^{N} G_i \frac{\omega^2 \tau_i^2}{1 + \omega^2 \tau_i^2}$$

$$G''(\omega) = \sum_{i=1}^{N} G_i \frac{\omega \tau_i}{1 + \omega^2 \tau_i^2}$$

## Learning Objectives

After completing this notebook, you will be able to:
- Convert relaxation modulus G(t) to dynamic moduli G'(ω), G''(ω)
- Convert dynamic moduli G'(ω), G''(ω) to relaxation modulus G(t)
- Understand NNLS fitting for non-negative Prony mode strengths
- Assess the effect of the number of Prony modes on conversion quality
- Apply domain interconversion to real rheological data

## Prerequisites

Basic understanding of:
- Linear viscoelasticity (LVE) and the relaxation modulus
- Dynamic mechanical analysis (DMA / DMTA)
- Rheological test modes (relaxation, oscillation)

**Estimated Time:** 25-35 minutes

In [ ]:
# Google Colab Setup - Run this cell first!
# Skip if running locally with rheojax already installed

import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Install rheojax and dependencies
    !pip install -q rheojax
    
    # Colab uses float32 by default - we need float64 for numerical stability
    # This MUST be set before importing JAX
    import os
    os.environ['JAX_ENABLE_X64'] = 'true'
    
    print("\u2713 RheoJAX installed successfully!")
    print("\u2713 Float64 precision enabled")

## Setup and Imports

In [ ]:
# Configure matplotlib for inline plotting in VS Code/Jupyter
# MUST come before importing matplotlib
%matplotlib inline

# Standard scientific computing imports
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from rheojax.core.data import RheoData
from rheojax.core.jax_config import safe_import_jax, verify_float64
from rheojax.transforms.prony_conversion import PronyConversion

# Safe JAX import
jax, jnp = safe_import_jax()
verify_float64()
print(f"\u2713 JAX float64 precision enabled")

# Reproducibility
np.random.seed(42)

# Plot configuration
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Suppress matplotlib backend warning in VS Code
warnings.filterwarnings('ignore', message='.*non-interactive.*')

## Theory: Prony Series in Linear Viscoelasticity

### Discrete Relaxation Spectrum

The Prony series is a **discrete approximation** to the continuous relaxation spectrum H(τ).
Each term G_i exp(-t/τ_i) represents a Maxwell element with spring constant G_i and
relaxation time τ_i. The equilibrium modulus G_e accounts for the elastic (solid-like) plateau.

### Analytical Domain Relations

The Prony series provides **exact analytical relations** between time and frequency domains:

| Domain | Formula |
|--------|--------|
| Time   | G(t) = G_e + Σ G_i exp(-t/τ_i) |
| Storage modulus | G'(ω) = G_e + Σ G_i ω²τ_i² / (1 + ω²τ_i²) |
| Loss modulus | G''(ω) = Σ G_i ωτ_i / (1 + ω²τ_i²) |

### NNLS Fitting Strategy

RheoJAX fits the Prony coefficients G_i using **Non-Negative Least Squares (NNLS)**:

1. Fix a set of log-spaced relaxation times τ_i spanning the data range
2. Build the kernel matrix A (analytical Prony basis functions)
3. Solve: minimize ‖A G_i - b‖₂ subject to G_i ≥ 0

The non-negativity constraint ensures **physically meaningful** mode strengths
(relaxation times cannot be negative).

### Applications

- **Domain interconversion**: Predict G'(ω) from G(t) or vice versa without re-testing
- **Model parameterization**: Warm-start for multi-mode Maxwell fitting
- **DMA ↔ stress relaxation**: Bridge different experimental protocols
- **Spectrum regularization**: Continuous relaxation spectrum approximation

## Part 1: Time → Frequency Conversion

Starting from relaxation modulus G(t) data, we convert to dynamic moduli G'(ω) and G''(ω).

In [ ]:
# Ground-truth 3-mode Maxwell model
G_modes = np.array([1000.0, 500.0, 200.0])   # Pa
tau_modes = np.array([10.0, 1.0, 0.1])        # s
G_e = 50.0                                    # Pa (equilibrium modulus)

# Relaxation time axis
t = np.logspace(-2, 2, 100)  # 0.01 to 100 s

# True G(t)
G_t = G_e + np.sum(G_modes * np.exp(-t[:, None] / tau_modes), axis=1)

# Add 2% Gaussian noise (realistic experimental scatter)
G_t_noisy = G_t * (1 + 0.02 * np.random.randn(len(t)))

print("Synthetic relaxation data (3-mode Maxwell + G_e):")
print(f"  G_modes: {G_modes} Pa")
print(f"  tau_modes: {tau_modes} s")
print(f"  G_e: {G_e} Pa")
print(f"  Time range: {t[0]:.2g} to {t[-1]:.2g} s ({len(t)} points)")
print(f"  Noise level: 2% relative")

# Wrap in RheoData
data_relax = RheoData(x=t, y=G_t_noisy, metadata={'test_mode': 'relaxation'})

In [ ]:
# Apply time → frequency conversion with 5 Prony modes
prony_t2f = PronyConversion(n_modes=5, direction="time_to_freq")
result, meta = prony_t2f._transform(data_relax)
prony_result = meta["prony_result"]

print(f"Prony fit summary:")
print(f"  Modes fitted: {prony_result.n_modes}")
print(f"  G_e (equilibrium): {prony_result.G_e:.2f} Pa")
print(f"  Mode strengths G_i (Pa): {np.round(prony_result.G_i, 2)}")
print(f"  Relaxation times tau_i (s): {np.round(prony_result.tau_i, 4)}")
print()
print(f"Converted frequency data:")
print(f"  Frequency range: {np.asarray(result.x)[0]:.3g} to {np.asarray(result.x)[-1]:.3g} rad/s")
print(f"  Output shape: {np.asarray(result.y).shape} (complex G*)")

In [ ]:
# Analytical frequency-domain prediction from true parameters
omega_true = np.asarray(result.x)
G_prime_analytical = G_e + np.sum(
    G_modes * (omega_true[:, None] * tau_modes)**2 / (1 + (omega_true[:, None] * tau_modes)**2),
    axis=1,
)
G_dprime_analytical = np.sum(
    G_modes * omega_true[:, None] * tau_modes / (1 + (omega_true[:, None] * tau_modes)**2),
    axis=1,
)

# Prony-converted moduli
G_star_prony = np.asarray(result.y)
G_prime_prony = G_star_prony.real
G_dprime_prony = G_star_prony.imag

# Prony reconstruction of G(t)
G_t_prony = prony_result.G_e + np.sum(
    prony_result.G_i * np.exp(-t[:, None] / prony_result.tau_i),
    axis=1,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: G(t) fit quality
ax = axes[0]
ax.loglog(t, G_t_noisy, 'o', markersize=4, alpha=0.4, color='gray', label='G(t) data (noisy)')
ax.loglog(t, G_t, '-', linewidth=2, color='blue', alpha=0.6, label='G(t) true')
ax.loglog(t, G_t_prony, '--', linewidth=2, color='red', label='Prony fit (5 modes)')
ax.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('G(t) (Pa)', fontsize=12, fontweight='bold')
ax.set_title('Step 1: Prony Fit to G(t)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

# Right: Resulting G'(omega), G''(omega)
ax = axes[1]
ax.loglog(omega_true, G_prime_analytical, '-', linewidth=2, color='blue', alpha=0.6, label="G' analytical")
ax.loglog(omega_true, G_dprime_analytical, '-', linewidth=2, color='red', alpha=0.6, label="G'' analytical")
ax.loglog(omega_true, G_prime_prony, '--', linewidth=2, color='blue', label="G' Prony")
ax.loglog(omega_true, G_dprime_prony, '--', linewidth=2, color='red', label="G'' Prony")
ax.set_xlabel('Frequency (rad/s)', fontsize=12, fontweight='bold')
ax.set_ylabel('G\', G\'\'(Pa)', fontsize=12, fontweight='bold')
ax.set_title('Step 2: Converted G\'(\u03c9), G\'\'(\u03c9)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
display(fig)
plt.close(fig)

# Error summary
rel_err_prime = np.mean(np.abs(G_prime_prony - G_prime_analytical) / G_prime_analytical) * 100
rel_err_dprime = np.mean(np.abs(G_dprime_prony - G_dprime_analytical) / G_dprime_analytical) * 100
print(f"Mean relative error G'(\u03c9): {rel_err_prime:.2f}%")
print(f"Mean relative error G''(\u03c9): {rel_err_dprime:.2f}%")

## Part 2: Frequency → Time Conversion

Starting from oscillation data G*(ω) = G'(ω) + i G''(ω), we convert to relaxation modulus G(t).

In [ ]:
# Generate oscillation data from the same 3-mode Maxwell model
omega = np.logspace(-2, 2, 80)  # 0.01 to 100 rad/s

G_prime = G_e + np.sum(
    G_modes * (omega[:, None] * tau_modes)**2 / (1 + (omega[:, None] * tau_modes)**2),
    axis=1,
)
G_dprime = np.sum(
    G_modes * omega[:, None] * tau_modes / (1 + (omega[:, None] * tau_modes)**2),
    axis=1,
)

# Combine as complex G* and wrap in RheoData
G_star = G_prime + 1j * G_dprime
data_osc = RheoData(x=omega, y=G_star, metadata={'test_mode': 'oscillation'})

print("Synthetic oscillation data (3-mode Maxwell + G_e):")
print(f"  Frequency range: {omega[0]:.2g} to {omega[-1]:.2g} rad/s ({len(omega)} points)")
print(f"  G' range: {G_prime.min():.1f} to {G_prime.max():.1f} Pa")
print(f"  G'' range: {G_dprime.min():.1f} to {G_dprime.max():.1f} Pa")
print(f"  Input dtype: {G_star.dtype} (complex G*)")

In [ ]:
# Apply frequency → time conversion with 5 Prony modes
prony_f2t = PronyConversion(n_modes=5, direction="freq_to_time")
result_time, meta_time = prony_f2t._transform(data_osc)
prony_result_f2t = meta_time["prony_result"]

print(f"Prony fit summary (freq \u2192 time):")
print(f"  Modes fitted: {prony_result_f2t.n_modes}")
print(f"  G_e (equilibrium): {prony_result_f2t.G_e:.2f} Pa")
print(f"  Mode strengths G_i (Pa): {np.round(prony_result_f2t.G_i, 2)}")
print(f"  Relaxation times tau_i (s): {np.round(prony_result_f2t.tau_i, 4)}")
print()
print(f"Converted time data:")
print(f"  Time range: {np.asarray(result_time.x)[0]:.3g} to {np.asarray(result_time.x)[-1]:.3g} s")
print(f"  Output shape: {np.asarray(result_time.y).shape}")

In [ ]:
# Converted time axis and G(t)
t_conv = np.asarray(result_time.x)
G_t_conv = np.asarray(result_time.y)

# Analytical G(t) at same time points
G_t_analytical = G_e + np.sum(
    G_modes * np.exp(-t_conv[:, None] / tau_modes), axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Input oscillation data
ax = axes[0]
ax.loglog(omega, G_prime, '-', linewidth=2, color='blue', label="G' input")
ax.loglog(omega, G_dprime, '-', linewidth=2, color='red', label="G'' input")
ax.set_xlabel('Frequency (rad/s)', fontsize=12, fontweight='bold')
ax.set_ylabel('G\', G\'\'(Pa)', fontsize=12, fontweight='bold')
ax.set_title('Input: Oscillation Data G*(\u03c9)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

# Right: Converted G(t)
ax = axes[1]
ax.loglog(t_conv, G_t_analytical, '-', linewidth=2.5, color='blue', alpha=0.7, label='G(t) analytical')
ax.loglog(t_conv, G_t_conv, '--', linewidth=2, color='red', label='G(t) Prony (5 modes)')
ax.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('G(t) (Pa)', fontsize=12, fontweight='bold')
ax.set_title('Output: Relaxation Modulus G(t)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
display(fig)
plt.close(fig)

rel_err_gt = np.mean(np.abs(G_t_conv - G_t_analytical) / G_t_analytical) * 100
print(f"Mean relative error G(t): {rel_err_gt:.2f}%")
print("\u2713 Frequency-to-time conversion successful!")

## Effect of Number of Modes

The number of Prony modes controls the resolution of the discrete relaxation spectrum.
Too few modes misses features; too many can overfit noisy data.

In [ ]:
# Compare n_modes = 3, 5, 10, 20 for time → frequency conversion
n_modes_list = [3, 5, 10, 20]
colors = ['steelblue', 'darkorange', 'green', 'purple']

# Analytical reference at fine frequency grid
omega_ref = np.logspace(-2, 2, 200)
G_prime_ref = G_e + np.sum(
    G_modes * (omega_ref[:, None] * tau_modes)**2 / (1 + (omega_ref[:, None] * tau_modes)**2),
    axis=1,
)
G_dprime_ref = np.sum(
    G_modes * omega_ref[:, None] * tau_modes / (1 + (omega_ref[:, None] * tau_modes)**2),
    axis=1,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot analytical reference
axes[0].loglog(omega_ref, G_prime_ref, 'k-', linewidth=3, alpha=0.4, label="G' analytical", zorder=10)
axes[1].loglog(omega_ref, G_dprime_ref, 'k-', linewidth=3, alpha=0.4, label="G'' analytical", zorder=10)

errors = {}
for n, color in zip(n_modes_list, colors):
    pc = PronyConversion(n_modes=n, direction="time_to_freq", omega_out=omega_ref)
    res, m = pc._transform(data_relax)
    G_star_n = np.asarray(res.y)
    G_p = G_star_n.real
    G_dp = G_star_n.imag

    rel_err = (
        np.mean(np.abs(G_p - G_prime_ref) / G_prime_ref)
        + np.mean(np.abs(G_dp - G_dprime_ref) / G_dprime_ref)
    ) / 2 * 100
    errors[n] = rel_err

    axes[0].loglog(omega_ref, G_p, '--', linewidth=1.5, color=color, label=f'n={n} (err={rel_err:.1f}%)')
    axes[1].loglog(omega_ref, G_dp, '--', linewidth=1.5, color=color, label=f'n={n} (err={rel_err:.1f}%)')

for ax, title, ylabel in zip(
    axes,
    ["G'(\u03c9) vs Number of Prony Modes", "G''(\u03c9) vs Number of Prony Modes"],
    ["G' (Pa)", "G'' (Pa)"],
):
    ax.set_xlabel('Frequency (rad/s)', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
display(fig)
plt.close(fig)

print("Convergence table (mean relative error across G', G''):")
print("-" * 40)
for n, err in errors.items():
    bar = '#' * int(err * 2)
    print(f"  n_modes={n:2d}: {err:5.2f}%  {bar}")
print("-" * 40)
print("Diminishing returns beyond ~10 modes for this dataset.")

## Key Takeaways

### Main Concepts

**1. Prony Series as a Domain Bridge:**
- A single Prony series parameterization simultaneously describes G(t), G'(ω), and G''(ω)
- Conversion is exact (no approximation error for the analytical relations)
- Fitting error comes only from approximating the true spectrum with N discrete modes

**2. NNLS Guarantees Physical Modes:**
- Mode strengths G_i are constrained to be non-negative
- Avoids unphysical oscillations that arise from unconstrained least squares
- Relaxation times τ_i are fixed on a log-spaced grid (only G_i are fitted)

**3. Mode Count Guidelines:**
- 3-5 modes: Captures broad features, robust for noisy data
- 5-10 modes: Good balance for typical rheological data
- 10-20 modes: High resolution; may overfit noisy data
- `n_modes=None` (auto): `max(3, min(N // 5, 20))` — sensible default

**4. Direction Matters for Input Format:**
- `time_to_freq`: Expects real G(t) data
- `freq_to_time`: Expects complex G*(ω) = G' + iG'' or (N, 2) array [G', G'']

**5. Practical Workflow:**
- Fit Prony series once; use for multiple downstream tasks
- Check `prony_result.G_i` for zero-weight modes (redundant τ_i)
- Validate conversion against analytical or reference data when possible

### Related Notebooks

- **[01-fft-analysis.ipynb](./01-fft-analysis.ipynb)**: Alternative frequency-domain analysis via FFT
- **[05-smooth-derivative.ipynb](./05-smooth-derivative.ipynb)**: Noise-robust differentiation (complementary interconversion)
- **[07-srfs-strain-rate-superposition.ipynb](./07-srfs-strain-rate-superposition.ipynb)**: Advanced viscoelastic transforms

### Advanced Topics

- Continuous relaxation spectrum inversion (Tikhonov regularization)
- Uncertainty propagation in Prony coefficients
- Fractional element extensions of the Prony series

---

## Session Information

In [ ]:
import sys

import scipy

import rheojax

print(f"Python version: {sys.version}")
print(f"RheoJAX version: {rheojax.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {scipy.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")